# Problem ID: Category_02_Problem01 — Flappy Bird Gap Aligner with Horizontal Scroll

### 1. Problem Scenario:
An autonomous flight agent operates in a 200x200 scrolling airspace. The agent starts at **(0, 100)** and must safely navigate through an incoming pipe wall featuring a clear passage zone (gap center at vertical position **100**). Pipes spawn at a horizontal distance of **200** and move toward the agent. The agent must stabilize its altitude within the target gap boundaries before crossing the threshold.

### 2. System Specifications:
- **State Space ($S$):** Discretized 2D grid matrix mapping horizontal distance to the pipe (`dx`) and relative vertical distance to the gap center (`dy`).
  - `dx`: 4 discrete intervals based on structural distance ($max(0, min(3, \lfloor pipe\_x / 50 \rfloor))$).
  - `dy`: 5 discrete elevation bands tracking vertical offset ($max(-2, min(2, \lfloor (bird\_y - gap\_y) / 20 \rfloor)) + 2$).
- **Action Space ($A$):** 2 discrete flight commands: `0: IDLE` (subject to gravitational descent of `+8` units), `1: FLAP` (subject to upward impulse of `-15` units).
- **Reward Function:** Pipe Clearance Success = `+20.0`, Boundary Crash / Pipe Collision = `-100.0`, Safe Survival Flight Step = `+0.1`.
- **Termination Rule:** The episode terminates with a major penalty if the agent impacts the lower/upper flight ceilings or fails to clear the vertical clearance threshold ($|bird\_y - gap\_y| \ge 25$) when the pipe coordinate passes `pipe_x <= 0`.

### 3. Student Implementation Deliverables:
1. **Task 1 (Cell 4):** Implement the $\epsilon$-Greedy flight action selection policy.
2. **Task 2 (Cell 5):** Implement the Tabular Q-Learning Bellman update rule matching the spatial grid indices:
   $$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$
3. **Execute & Verify (Cells 6 & 7):** Run the simulation driver, validating that structural pipe clearance rewards converge and overall episode survival length increases.

In [ ]:
# Cell 2: Standard Libraries & Environment Parameters (LOCKED)
import numpy as np
import matplotlib.pyplot as plt
import random

# Set deterministic seed for reproducible results
np.random.seed(42)
random.seed(42)

# Environment Spatial Bounds
AIRSPACE_HEIGHT = 200
START_PIPE_X = 200
GAP_CENTER_Y = 100

# Reinforcement Learning & Simulation Parameters
NUM_EPISODES = 500
MAX_STEPS_PER_EPISODE = 100
ALPHA = 0.1       # Learning Rate
GAMMA = 0.95      # Discount Factor
EPSILON = 0.1     # Exploration Rate

In [ ]:
# Cell 3: Environment Simulator Scaffold (LOCKED)
class FlappyGapEnv:
    def __init__(self):
        self.reset()
        
    def reset(self):
        """Resets bird and pipe properties, returning the initial discretized state tuple."""
        self.bird_y = 100
        self.pipe_x = START_PIPE_X
        self.gap_y = GAP_CENTER_Y
        return self._get_state()
        
    def _get_state(self):
        # Quantize horizontal distance to incoming pipe (4 bins)
        dx = max(0, min(3, int(self.pipe_x / 50)))
        # Quantize vertical offset relative to gap center (5 bins)
        dy = max(-2, min(2, int((self.bird_y - self.gap_y) / 20))) + 2
        return (dx, dy)
        
    def step(self, action):
        """
        Executes action: 0: IDLE (FALL), 1: FLAP (JUMP)
        Returns: (next_state, reward, done)
        """
        if action == 1:
            self.bird_y -= 15  # Upward impulse
        else:
            self.bird_y += 8   # Gravitational fall
            
        self.pipe_x -= 10      # Airspace scroll speed
        
        state = self._get_state()
        
        # Check boundary collision
        if self.bird_y < 0 or self.bird_y > AIRSPACE_HEIGHT:
            return state, -100.0, True
            
        # Check pipe threshold crossing
        if self.pipe_x <= 0:
            if abs(self.bird_y - self.gap_y) < 25:
                return state, 20.0, True  # Successful clearance
            else:
                return state, -100.0, True # Structural crash
                
        return state, 0.1, False  # Survival reward

In [ ]:
# Cell 4: [TASK 1] Agent Class & Action Selection (STUDENT IMPLEMENTATION)
class FlappyQLearningAgent:
    def __init__(self, alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON):
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        # Dimensions matching the state discretization grid: 4 x 5 x 2
        self.q_table = np.zeros((4, 5, 2))
        
    def choose_action(self, state):
        """
        TASK 1: Implement Epsilon-Greedy Action Selection Policy.

        Requirements:
        - Generate a random number in [0, 1).
        - If random value < self.epsilon, return a random flight command (0 or 1).
        - Otherwise, return the action index with the maximum Q-value 
          from the Q-table for the current spatial state index.
        """
        ### >>> YOUR CODE HERE <<< ###
        pass
        ### ========================== ###

In [ ]:
# Cell 5: [TASK 2] Q-Learning Bellman Update (STUDENT IMPLEMENTATION)
def update_agent(self, state, action, reward, next_state, done):
    """
    TASK 2: Implement the Tabular Q-Learning Update Rule.

    Requirements:
    - If done is True: target = reward
    - Otherwise:
        target = reward + self.gamma * max(Q(next_state, all_actions))
    - Update:
        Q(state, action) = Q(state, action) + self.alpha * (target - Q(state, action))

    The function is attached to FlappyQLearningAgent below so that the
    notebook retains exactly seven cells.
    """
    ### >>> YOUR CODE HERE <<< ###
    pass
    ### ========================== ###

FlappyQLearningAgent.update_agent = update_agent

In [ ]:
# Cell 6: Simulation Execution & Test Driver (LOCKED)
env = FlappyGapEnv()
agent = FlappyQLearningAgent()

rewards_history = []
steps_history = []

for ep in range(NUM_EPISODES):
    state = env.reset()
    total_reward = 0
    step_count = 0
    done = False

    while not done and step_count < MAX_STEPS_PER_EPISODE:
        action = agent.choose_action(state)
        next_state, reward, done = env.step(action)
        agent.update_agent(state, action, reward, next_state, done)

        state = next_state
        total_reward += reward
        step_count += 1

    rewards_history.append(total_reward)
    steps_history.append(step_count)

print(f"Simulation completed across {NUM_EPISODES} episodes.")
print(f"Average Reward (Last 50 Episodes): {np.mean(rewards_history[-50:]):.2f}")
print(f"Average Steps to Goal (Last 50 Episodes): {np.mean(steps_history[-50:]):.2f}")

In [ ]:
# Cell 7: Performance Metric Visualization (LOCKED)
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(rewards_history, color='tab:blue', alpha=0.7)
plt.title("Reward Trajectory per Episode")
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(steps_history, color='tab:orange', alpha=0.7)
plt.title("Flight Lifetime per Episode")
plt.xlabel("Episode")
plt.ylabel("Steps Survived")
plt.grid(True)

plt.tight_layout()
plt.show()